# Predictive validity in two tracks

This study measures prediction of a continuous response under smooth, interaction and factor-dependent mean functions. The known-specification track supplies the true term structure and Gaussian family to mboostrs. The discovery track supplies only the data and response name to `mb.discover`. CatBoost, LightGBM, XGBoost and scikit-learn histogram boosting receive data alone and tune four configurations each. The table reports results after the library changes described below, with the baseline and the paired changes retained. The results apply to the mboostrs build described in the reproduction section.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from great_tables import GT
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(ROOT))
from benches.report_predictive_tracks import DIRECTORY, gap_table, headline

baseline = json.loads((DIRECTORY / "baseline/summary.json").read_text())
after = json.loads((DIRECTORY / "after/summary.json").read_text())
display(headline(after))


GT(_tbl_data=                  track             method    smooth  smooth_se  interaction  \
0   known_specification     mboostrs_known  1.005814   0.007371     1.045460   
1   known_specification           catboost  1.075020   0.005724     1.151736   
2   known_specification           lightgbm  1.149116   0.012020     1.265260   
3   known_specification            xgboost  1.095860   0.008578     1.386397   
4   known_specification             histgb  1.151703   0.012221     1.260903   
5   known_specification              ridge  1.784817   0.010096     2.564508   
6             discovery  mboostrs_discover  1.008951   0.007894     1.122592   
7             discovery           catboost  1.075020   0.005724     1.151736   
8             discovery           lightgbm  1.149116   0.012020     1.265260   
9             discovery            xgboost  1.095860   0.008578     1.386397   
10            discovery             histgb  1.151703   0.012221     1.260903   
11            discovery              ridge  1.784817   0.010096     2.564508   

    interaction_se    factor  factor_se  
0         0.012736  0.995122   0.011696  
1         0.018983  1.155387   0.025569  
2         0.022642  1.409864   0.017759  
3         0.022692  1.732358   0.106627  
4         0.025025  1.409704   0.013795  
5         0.031397  2.987684   0.027749  
6         0.020839  1.056597   0.010476  
7         0.018983  1.155387   0.025569  
8         0.022642  1.409864   0.017759  
9         0.022692  1.732358   0.106627  
10        0.025025  1.409704   0.013795  
11        0.031397  2.987684   0.027749  , _body=<great_tables._gt_data.Body object at 0x11201fce0>, _boxhead=Boxhead([ColInfo(var='track', type=<ColInfoTypeEnum.row_group: 3>, column_label='track', column_align='left', column_width=None), ColInfo(var='method', type=<ColInfoTypeEnum.stub: 2>, column_label='method', column_align='left', column_width=None), ColInfo(var='smooth', type=<ColInfoTypeEnum.default: 1>, column_label='MSE', column_align='right', column_width=None), ColInfo(var='smooth_se', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None), ColInfo(var='interaction', type=<ColInfoTypeEnum.default: 1>, column_label='MSE', column_align='right', column_width=None), ColInfo(var='interaction_se', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None), ColInfo(var='factor', type=<ColInfoTypeEnum.default: 1>, column_label='MSE', column_align='right', column_width=None), ColInfo(var='factor_se', type=<ColInfoTypeEnum.default: 1>, column_label='SE', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10f1209e0>, _spanners=Spanners([SpannerInfo(spanner_id='Smooth', spanner_level=0, spanner_label='Smooth', spanner_units=None, spanner_pattern=None, vars=['smooth', 'smooth_se'], built=None), SpannerInfo(spanner_id='Interaction', spanner_level=0, spanner_label='Interaction', spanner_units=None, spanner_pattern=None, vars=['interaction', 'interaction_se'], built=None), SpannerInfo(spanner_id='Factor', spanner_level=0, spanner_label='Factor', spanner_units=None, spanner_pattern=None, vars=['factor', 'factor_se'], built=None)]), _heading=Heading(title='Predictive validity: every method in both information tracks', subtitle='Outer-test MSE and standard error across eight independent draws; lower is better within a process', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x112090710>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x112090740>, _source_notes=['Every MSE is in squared response units. SE is the sample SD divided by sqrt(8); failed/incomplete cells are absent, not successful-only means.', 'Competitors receive data alone and share their identical predictions across tracks. Four configurations are validation-tuned per competitor; Ridge is the prespecified simple reference. mboostrs uses five internal folds and a diffe

## Information tracks

Each of eight independent draws per process has 800 development rows and
2,000 independent outer-test rows. Competitors train on the first 600
development rows, choose settings by MSE on the next 200, and refit the
selected configuration on all 800. Both mboostrs tracks use all 800 rows
with five internal folds. Competitor predictions are identical across the
two track reports because competitors receive the same data and no structure.
Their tuning design and fit counts differ from the full discovery procedure;
this study does not establish matched-compute superiority.

All five numeric predictors are independent Uniform(−1,1). The factor process
adds an independent two-level label. Noise is independent Gaussian with
standard deviation 1 response unit. The generating means are
`2*sin(pi*x1)+0.5*x2`, `2*sin(pi*x1*x2)+0.5*x3`, and
`2*sin(pi*x1)*(+1 for g=a, −1 for g=b)+0.5*x2`. The other columns are noise.
The [frozen protocol](predictive_tracks/v2/PROTOCOL.md) gives the supplied
formulas, all configuration grids, seeds, selection rules and score anchors.
The true conditional mean is an oracle reference only and selects no settings.

The known factor formula contains a baseline `bbs(x1, df=4)` plus
`bbs(x1, df=4, by=g)`. Treatment coding makes the latter a difference from the
reference level. Together they represent both level curves. These supplied
terms receive no discovery credit; D1a and D1b are not scored here.


## Paired competitor gaps

CatBoost has the lowest mean test MSE among the four tuned tree competitors
on all three processes. That choice uses test results and is a retrospective
best. The primary gap is the mean of the eight paired loss ratios, rather
than the ratio of two mean losses. Its SE is the sample standard deviation of
those ratios divided by the square root of eight. It treats the selected
competitor as fixed and omits winner-selection uncertainty; it is a
descriptive Monte Carlo SE, not a general confidence guarantee.

Known-specification gaps are −6.43% ± 0.72 points for smooth effects,
−9.16% ± 0.77 for the pure interaction and −13.69% ± 1.43 for factor curves.
Discovery gaps after the library changes are −6.13% ± 0.78, −2.54% ± 0.56 and
−8.23% ± 2.25 points, respectively. Every ± quantity is one SE. All six point
gaps meet the at-most-1% target on this focused battery. The result is limited
to these generators, 800-row development sets and finite competitor grids.


In [2]:
display(gap_table(after))


GT(_tbl_data=                 track         task best_competitor  completed    gap_pct  \
0  known_specification       smooth        catboost          8  -6.426238   
1            discovery       smooth        catboost          8  -6.133762   
2  known_specification  interaction        catboost          8  -9.162045   
3            discovery  interaction        catboost          8  -2.544900   
4  known_specification       factor        catboost          8 -13.688240   
5            discovery       factor        catboost          8  -8.232909   

   gap_se_pct  difference_mse  difference_se  within_1pct  unresolved_at_2se  
0    0.718655       -0.069206       0.007951         True              False  
1    0.775065       -0.066069       0.008552         True              False  
2    0.768553       -0.106276       0.009756         True              False  
3    0.557869       -0.029145       0.006463         True              False  
4    1.434897       -0.160265       0.021006         True              False  
5    2.248840       -0.098790       0.029965         True              False  , _body=<great_tables._gt_data.Body object at 0x11176b9e0>, _boxhead=Boxhead([ColInfo(var='track', type=<ColInfoTypeEnum.row_group: 3>, column_label='track', column_align='left', column_width=None), ColInfo(var='task', type=<ColInfoTypeEnum.stub: 2>, column_label='task', column_align='left', column_width=None), ColInfo(var='best_competitor', type=<ColInfoTypeEnum.default: 1>, column_label='best_competitor', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='gap_pct', type=<ColInfoTypeEnum.default: 1>, column_label='gap_pct', column_align='right', column_width=None), ColInfo(var='gap_se_pct', type=<ColInfoTypeEnum.default: 1>, column_label='gap_se_pct', column_align='right', column_width=None), ColInfo(var='difference_mse', type=<ColInfoTypeEnum.default: 1>, column_label='difference_mse', column_align='right', column_width=None), ColInfo(var='difference_se', type=<ColInfoTypeEnum.default: 1>, column_label='difference_se', column_align='right', column_width=None), ColInfo(var='within_1pct', type=<ColInfoTypeEnum.default: 1>, column_label='within_1pct', column_align='center', column_width=None), ColInfo(var='unresolved_at_2se', type=<ColInfoTypeEnum.default: 1>, column_label='unresolved_at_2se', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10ccc5130>, _spanners=Spanners([SpannerInfo(spanner_id='Gap (%)', spanner_level=0, spanner_label='Gap (%)', spanner_units=None, spanner_pattern=None, vars=['gap_pct', 'gap_se_pct'], built=None), SpannerInfo(spanner_id='Paired MSE difference', spanner_level=0, spanner_label='Paired MSE difference', spanner_units=None, spanner_pattern=None, vars=['difference_mse', 'difference_se'], built=None)]), _heading=Heading(title='Paired gap to the strongest complete named competitor', subtitle='Percent gap and paired Monte Carlo SE; negative favors mboostrs; target at most +1%', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x112090380>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1120903e0>, _source_notes=['The competitor is selected retrospectively from mean test MSE. SE treats this selected winner as fixed and does not include method-selection uncertainty; it is descriptive.', 'Each row uses all eight matching draws. A point gap above 1% is filed even when within two SE of zero. Tracks and processes are never pooled.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1120905c0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1118338c0>, <great_tables._gt_data.FormatInfo object at 0x112092de0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), tabl

## Factor-reference dictionary

The baseline completed all 168 planned method/draw jobs, but factor
discovery was 75.14% ± 4.14 percentage points behind CatBoost. All eight
discovered factor formulas omitted a baseline curve for `x1`. The marginal
mean of that predictor cancels between the two groups, so elimination can
remove it. An adopted `bbs(x1, by=g)` then models only a difference from the
reference level and is zero on that level.

An independent diagnostic gives mean squared error against the true curve of
2.085 for the reference group and 0.007 for the other group. The change makes
a baseline smooth available in the factor interaction comparison when needed,
preserves its input scale and the surviving dictionary, and records the
comparison's reference terms. The adoption threshold is unchanged.

With the revised factor dictionary (the set of candidate terms that discovery may adopt), factor discovery mean MSE falls from 2.0169 to
1.0566. The paired change is −0.9603 ± 0.0232 squared response units, or
−47.58% ± 0.73 percentage points. All eight factor draws improve. The 120
competitor and ridge-regression (a regularised linear baseline) predictions and all 40 mboostrs predictions outside the
factor-discovery cell are byte-identical between phases.


In [3]:
paired = json.loads((DIRECTORY / "paired_improvement.json").read_text())
changes = pd.DataFrame(paired["changes"])
display(GT(changes, rowname_col="task", groupname_col="track")
    .tab_header(title="Same benchmark before and after the proposed repairs",
        subtitle="Test MSE and paired change across eight draws; negative change improves prediction")
    .tab_spanner("Prediction", ["before_mse", "after_mse"])
    .tab_spanner("Paired change", ["change_mse", "change_se", "change_pct", "change_pct_se"])
    .fmt_number(columns=["before_mse", "after_mse", "change_mse", "change_se"], decimals=4)
    .fmt_number(columns=["change_pct", "change_pct_se"], decimals=2)
    .tab_source_note("SE is the sample SD of paired changes divided by sqrt(8). The protocol, seeds and all competitor settings are unchanged.")
    .tab_source_note("The repairs affect dictionary construction and predictor metadata. Fitting and CV use the existing Rust-backed routines."))


GT(_tbl_data=          task                track  completed  before_mse  after_mse  \
0       smooth  known_specification          8    1.005814   1.005814   
1       smooth            discovery          8    1.008951   1.008951   
2  interaction  known_specification          8    1.045460   1.045460   
3  interaction            discovery          8    1.122592   1.122592   
4       factor  known_specification          8    0.995122   0.995122   
5       factor            discovery          8    2.016918   1.056597   

   change_mse  change_se  change_pct  change_pct_se  unchanged_predictions  
0    0.000000   0.000000    0.000000       0.000000                      8  
1    0.000000   0.000000    0.000000       0.000000                      8  
2    0.000000   0.000000    0.000000       0.000000                      8  
3    0.000000   0.000000    0.000000       0.000000                      8  
4    0.000000   0.000000    0.000000       0.000000                      8  
5   -0.960321   0.023167  -47.575886       0.731942                      0  , _body=<great_tables._gt_data.Body object at 0x1105bffe0>, _boxhead=Boxhead([ColInfo(var='task', type=<ColInfoTypeEnum.stub: 2>, column_label='task', column_align='left', column_width=None), ColInfo(var='track', type=<ColInfoTypeEnum.row_group: 3>, column_label='track', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='before_mse', type=<ColInfoTypeEnum.default: 1>, column_label='before_mse', column_align='right', column_width=None), ColInfo(var='after_mse', type=<ColInfoTypeEnum.default: 1>, column_label='after_mse', column_align='right', column_width=None), ColInfo(var='change_mse', type=<ColInfoTypeEnum.default: 1>, column_label='change_mse', column_align='right', column_width=None), ColInfo(var='change_se', type=<ColInfoTypeEnum.default: 1>, column_label='change_se', column_align='right', column_width=None), ColInfo(var='change_pct', type=<ColInfoTypeEnum.default: 1>, column_label='change_pct', column_align='right', column_width=None), ColInfo(var='change_pct_se', type=<ColInfoTypeEnum.default: 1>, column_label='change_pct_se', column_align='right', column_width=None), ColInfo(var='unchanged_predictions', type=<ColInfoTypeEnum.default: 1>, column_label='unchanged_predictions', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1120911f0>, _spanners=Spanners([SpannerInfo(spanner_id='Prediction', spanner_level=0, spanner_label='Prediction', spanner_units=None, spanner_pattern=None, vars=['before_mse', 'after_mse'], built=None), SpannerInfo(spanner_id='Paired change', spanner_level=0, spanner_label='Paired change', spanner_units=None, spanner_pattern=None, vars=['change_mse', 'change_se', 'change_pct', 'change_pct_se'], built=None)]), _heading=Heading(title='Same benchmark before and after the proposed repairs', subtitle='Test MSE and paired change across eight draws; negative change improves prediction', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x112092b70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x112092360>, _source_notes=['SE is the sample SD of paired changes divided by sqrt(8). The protocol, seeds and all competitor settings are unchanged.', 'The repairs affect dictionary construction and predictor metadata. Fitting and CV use the existing Rust-backed routines.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x112092ea0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1117f7860>, <great_tables._gt_data.FormatInfo object at 0x112092690>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True,

In [4]:
factor_pairs = [g for g in baseline["gaps"] if g["task"] == "factor" and g["track"] == "discovery"][0]["paired"]
after_pairs = [g for g in after["gaps"] if g["task"] == "factor" and g["track"] == "discovery"][0]["paired"]
rows = pd.DataFrame([
    {"seed": b["seed"], "before_mse": b["mboost_mse"], "after_mse": a["mboost_mse"],
     "catboost_mse": b["competitor_mse"], "after_gap_pct": a["gap_pct"]}
    for b, a in zip(factor_pairs, after_pairs, strict=True)
])
display(GT(rows, rowname_col="seed")
    .tab_header(title="Discovery track: every factor draw",
        subtitle="Outer-test MSE and gap percent to retrospective CatBoost; lower is better")
    .tab_spanner("Test MSE", ["before_mse", "after_mse", "catboost_mse"])
    .fmt_number(columns=["before_mse", "after_mse", "catboost_mse"], decimals=4)
    .fmt_number(columns="after_gap_pct", decimals=2)
    .tab_source_note("Each row shares development and test observations across all methods and phases. No draw is omitted."))


GT(_tbl_data=   seed  before_mse  after_mse  catboost_mse  after_gap_pct
0     0    1.949621   1.057371      1.100228      -3.895270
1     1    1.991195   1.077351      1.149198      -6.251923
2     2    2.053258   1.022341      1.136325     -10.030903
3     3    2.018133   1.116310      1.118055      -0.156020
4     4    1.999058   1.037396      1.129436      -8.149235
5     5    2.013153   1.045065      1.325758     -21.172247
6     6    1.970156   1.035405      1.171355     -11.606174
7     7    2.140768   1.061536      1.112738      -4.601501, _body=<great_tables._gt_data.Body object at 0x111f8faa0>, _boxhead=Boxhead([ColInfo(var='seed', type=<ColInfoTypeEnum.stub: 2>, column_label='seed', column_align='right', column_width=None), ColInfo(var='before_mse', type=<ColInfoTypeEnum.default: 1>, column_label='before_mse', column_align='right', column_width=None), ColInfo(var='after_mse', type=<ColInfoTypeEnum.default: 1>, column_label='after_mse', column_align='right', column_width=None), ColInfo(var='catboost_mse', type=<ColInfoTypeEnum.default: 1>, column_label='catboost_mse', column_align='right', column_width=None), ColInfo(var='after_gap_pct', type=<ColInfoTypeEnum.default: 1>, column_label='after_gap_pct', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x112092c00>, _spanners=Spanners([SpannerInfo(spanner_id='Test MSE', spanner_level=0, spanner_label='Test MSE', spanner_units=None, spanner_pattern=None, vars=['before_mse', 'after_mse', 'catboost_mse'], built=None)]), _heading=Heading(title='Discovery track: every factor draw', subtitle='Outer-test MSE and gap percent to retrospective CatBoost; lower is better', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x112090980>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x112093560>, _source_notes=['Each row shares development and test observations across all methods and phases. No draw is omitted.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x112093b90>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10ccc7a70>, <great_tables._gt_data.FormatInfo object at 0x112093830>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value

## Fresh controls

A separate [frozen protocol](factor_reference/v2/PROTOCOL.md) uses six
new independent draws in each of seven control families. All 84 phase/draw
jobs complete. Every family's mean after/before MSE ratio is at most 1.01;
improvements do not compensate for a failing family. Opposite factor curves,
relabelled factors and log-scale curves improve by about 74–77%. Smooth,
null, numeric-indicator and additive-factor predictions remain byte-identical.
The null interaction count does not increase, and both opposed-label families
retain nonconstant reference-level predictions.

This is a non-regression check against the parent build, which already contains the schema change, and so isolates the dictionary change. It does not replace the competitor comparison
or establish calibrated false-discovery control. 


In [5]:
controls_path = ROOT / "docs/book/benchmarks/factor_reference/v2/summary.json"
controls = json.loads(controls_path.read_text())
display(GT(pd.DataFrame(controls["families"]), rowname_col="case")
    .tab_header(title="Discovery repair: fresh control families",
        subtitle="Test MSE, after/before ratio and paired change SE; lower is better; ratio limit 1.01")
    .tab_spanner("Prediction", ["before_mse", "after_mse", "ratio"])
    .tab_spanner("Paired change", ["change", "change_se"])
    .fmt_number(columns=["before_mse", "after_mse", "ratio", "change", "change_se"], decimals=4)
    .tab_source_note("Six independent new draws per family; SE is SD of paired changes divided by sqrt(6). Every family must pass.")
    .tab_source_note("This non-regression gate has separate numeric-prediction, null-adoption and reference-curve checks; it is not a best-competitor or FDR guarantee."))


GT(_tbl_data=          case  completed  before_mse  after_mse     ratio    change  \
0      opposed          6    1.283487   0.323959  0.252405 -0.959528   
1   relabelled          6    1.286206   0.332899  0.258823 -0.953307   
2       smooth          6    0.254664   0.254664  1.000000  0.000000   
3         null          6    0.248753   0.248753  1.000000  0.000000   
4      numeric          6    0.277866   0.277866  1.000000  0.000000   
5  factor_main          6    0.255426   0.255426  1.000000  0.000000   
6       logged          6    1.314244   0.306162  0.232957 -1.008082   

   change_se  passes_1_01  
0   0.022522         True  
1   0.020812         True  
2   0.000000         True  
3   0.000000         True  
4   0.000000         True  
5   0.000000         True  
6   0.016301         True  , _body=<great_tables._gt_data.Body object at 0x10c3c7e60>, _boxhead=Boxhead([ColInfo(var='case', type=<ColInfoTypeEnum.stub: 2>, column_label='case', column_align='left', column_width=None), ColInfo(var='completed', type=<ColInfoTypeEnum.default: 1>, column_label='completed', column_align='right', column_width=None), ColInfo(var='before_mse', type=<ColInfoTypeEnum.default: 1>, column_label='before_mse', column_align='right', column_width=None), ColInfo(var='after_mse', type=<ColInfoTypeEnum.default: 1>, column_label='after_mse', column_align='right', column_width=None), ColInfo(var='ratio', type=<ColInfoTypeEnum.default: 1>, column_label='ratio', column_align='right', column_width=None), ColInfo(var='change', type=<ColInfoTypeEnum.default: 1>, column_label='change', column_align='right', column_width=None), ColInfo(var='change_se', type=<ColInfoTypeEnum.default: 1>, column_label='change_se', column_align='right', column_width=None), ColInfo(var='passes_1_01', type=<ColInfoTypeEnum.default: 1>, column_label='passes_1_01', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x112093260>, _spanners=Spanners([SpannerInfo(spanner_id='Prediction', spanner_level=0, spanner_label='Prediction', spanner_units=None, spanner_pattern=None, vars=['before_mse', 'after_mse', 'ratio'], built=None), SpannerInfo(spanner_id='Paired change', spanner_level=0, spanner_label='Paired change', spanner_units=None, spanner_pattern=None, vars=['change', 'change_se'], built=None)]), _heading=Heading(title='Discovery repair: fresh control families', subtitle='Test MSE, after/before ratio and paired change SE; lower is better; ratio limit 1.01', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1120930e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x112090c80>, _source_notes=['Six independent new draws per family; SE is SD of paired changes divided by sqrt(6). Every family must pass.', 'This non-regression gate has separate numeric-prediction, null-adoption and reference-curve checks; it is not a best-competitor or FDR guarantee.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x112092f90>, _formats=[<great_tables._gt_data.FormatInfo object at 0x112093020>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto'

## Composed predictor recipes

A separate reporting failure concerned composed predictors. A lowered factor-by term refers to a generated design,
which in turn reads a transformed column such as `log__x`. Reporting resolved
only one layer and asked the caller to supply that internal name. A model
could fit and predict correctly, then fail while discovery described it.

The schema change follows the recipe back to original inputs, preserves their
order, deduplicates shared sources and rejects cyclic metadata. On the same
42 original controls it resolves all three failures, while every one of the
39 successful prediction digests is unchanged. This measures
restored availability; no before/after numerical improvement is assigned to
rows whose baseline crashed. It is separate from the factor candidate dictionary change.


## Scorecard profile

D11 (predictive validity) scores 3 separately on all three processes in both tracks: outer
validation improves on the prespecified ridge-regression reference by more than twice
the paired SE, with all failures and uncertainty reported. The anchor rule
is fixed by the protocol. A D11 score of 3 measures useful
outer prediction; it does not by itself mean the 1% competitive target was
met. The baseline factor-discovery cell also scored 3 while losing to CatBoost.

Levels 4 and 5 are unavailable under these unequal search designs and this
focused battery. The remaining eleven dimensions are unmeasured. In particular,
predictive loss does not grade recovery, causal interpretation, report quality,
interval calibration or the speed of an analyst session. The profiles have no
aggregate and are not averaged with earlier studies.


In [6]:
card = json.loads((DIRECTORY / "after/scorecard.json").read_text())
profiles = card["profiles"]
rows = []
for i, dimension in enumerate(profiles[0]["dimensions"]):
    row = {"dimension": f"{dimension['id']} {dimension['name']}"}
    for profile in profiles:
        d = profile["dimensions"][i]
        row[f"{profile['track']} / {profile['task']}"] = str(d["score"]) if d["score"] is not None else "unmeasured"
    rows.append(row)
display(GT(pd.DataFrame(rows), rowname_col="dimension")
    .tab_header(title="Scorecard 2.1: separate predictive-validity profiles",
        subtitle="Anchored scores 1–5, higher is better; unmeasured dimensions receive no numeric score")
    .tab_source_note("Each column is one track/process profile. Only D11 is measured; no aggregate is defined."))


GT(_tbl_data=                                            dimension  \
0   D1a Relationship recovery under a prescribed s...   
1          D1b Relationship discovery from data alone   
2                      D2 Causal humility and support   
3                              D3 Explanation quality   
4                        D4 Importance quantification   
5                     D5 Experimental iteration speed   
6                     D6 Robustness across conditions   
7                               D7 Human-friendliness   
8                               D8 Agent-friendliness   
9                D9 Uncertainty honesty (calibration)   
10               D10 Reproducibility and auditability   
11                            D11 Predictive validity   

   known_specification / smooth discovery / smooth  \
0                    unmeasured         unmeasured   
1                    unmeasured         unmeasured   
2                    unmeasured         unmeasured   
3                    unmeasured         unmeasured   
4                    unmeasured         unmeasured   
5                    unmeasured         unmeasured   
6                    unmeasured         unmeasured   
7                    unmeasured         unmeasured   
8                    unmeasured         unmeasured   
9                    unmeasured         unmeasured   
10                   unmeasured         unmeasured   
11                            3                  3   

   known_specification / interaction discovery / interaction  \
0                         unmeasured              unmeasured   
1                         unmeasured              unmeasured   
2                         unmeasured              unmeasured   
3                         unmeasured              unmeasured   
4                         unmeasured              unmeasured   
5                         unmeasured              unmeasured   
6                         unmeasured              unmeasured   
7                         unmeasured              unmeasured   
8                         unmeasured              unmeasured   
9                         unmeasured              unmeasured   
10                        unmeasured              unmeasured   
11                                 3                       3   

   known_specification / factor discovery / factor  
0                    unmeasured         unmeasured  
1                    unmeasured         unmeasured  
2                    unmeasured         unmeasured  
3                    unmeasured         unmeasured  
4                    unmeasured         unmeasured  
5                    unmeasured         unmeasured  
6                    unmeasured         unmeasured  
7                    unmeasured         unmeasured  
8                    unmeasured         unmeasured  
9                    unmeasured         unmeasured  
10                   unmeasured         unmeasured  
11                            3                  3  , _body=<great_tables._gt_data.Body object at 0x11201fce0>, _boxhead=Boxhead([ColInfo(var='dimension', type=<ColInfoTypeEnum.stub: 2>, column_label='dimension', column_align='left', column_width=None), ColInfo(var='known_specification / smooth', type=<ColInfoTypeEnum.default: 1>, column_label='known_specification / smooth', column_align='left', column_width=None), ColInfo(var='discovery / smooth', type=<ColInfoTypeEnum.default: 1>, column_label='discovery / smooth', column_align='left', column_width=None), ColInfo(var='known_specification / interaction', type=<ColInfoTypeEnum.default: 1>, column_label='known_specification / interaction', column_align='left', column_width=None), ColInfo(var='discovery / interaction', type=<ColInfoTypeEnum.default: 1>, column_label='discovery / interaction', column_align='left', column_width=None), ColInfo(var='known_specification / factor', type=<ColInfoTypeEnum.default: 1>, column_label='known_specification / factor', column_align='left', column_width=None), ColInfo(var='d

## Known-specification formula

An earlier version of the known-specification factor formula omitted the reference-level curve, so its results are not used here.


## Reproduction and limits

The [baseline](predictive_tracks/v2/baseline.json) and
[after](predictive_tracks/v2/after.json) artifacts retain all 168 jobs each,
input/prediction fingerprints, configurations, complete discovery logs,
versions and loaded Python/native identities. The runner's source checkout is
separate from the imported library checkout; the
[baseline library](predictive_tracks/v2/baseline_library.json) and
[fixed library](predictive_tracks/v2/after_library.json) records identify both.
The native extension was built in release mode. The changes add no Rust code.

The [baseline fold audit](predictive_tracks/v2/baseline_folds.json) and
[after fold audit](predictive_tracks/v2/after_folds.json) independently repeat
each of the 48 mboostrs fits per phase, observe actual Python resampling
returns and verify exact prediction/MSE replay. Rust-internal screening
half-splits remain defined by the frozen code and seeds. This supplementary
instrumentation selects no settings and earns no additional scorecard grade.

Run `python -m benches.bench_predictive_tracks --phase baseline` with the
recorded parent library on `PYTHONPATH`, then `--phase after` with the fixed
library.  Run
`python -m benches.report_predictive_tracks --phase after` to rebuild the
assessment, and `python -m benches.capture_predictive_folds --phase after`
for the supplemental audit. Repeating the before report uses `--phase baseline`.
The [validation record](predictive_tracks/v2/VALIDATION.md) gives tests and
reproduction limits. 

All runs use one thread and serial jobs within a phase. Shared-hardware timing
is descriptive. The 1% point-gap target is met on this study's six cells, while
other dimensions, held-back generator families, small samples, dependence and
distribution shift remain outside these measurements. 

The factor in this study is a string label. A separate exploratory numeric
`0/1` encoding still loses the reference curve. That diagnosis is
outside this frozen competitor battery; the measured 1% result does not
establish invariance to predictor encoding.
